#### This notebook is used to experiment with DTGraph rules and transformations.

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
def reload_dtgraph():
    import importlib
    import dtgraph.parser
    import dtgraph.rule

    importlib.reload(dtgraph.parser)
    importlib.reload(dtgraph.rule)

reload_dtgraph()

In [4]:
from dtgraph.scenarios.movies import Movies

Movies.load(graph)

Flushed database: Deleted 273 nodes, deleted 977 relationships, completed after 756 ms.
Load scenario: Added 171 labels, created 171 nodes, set 564 properties, created 253 relationships, completed after 3673 ms.


### Node Rules 

In [ ]:
generate_films = Rule('''
MATCH (m:Movie)
GENERATE
(x = (x,z):Film {
    title = m.title
})
''')


complex_pattern1 = Rule('''
MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(b:Person)
GENERATE
(x = (a):Actor {
    name = a.name,
    experienced = a.born < 1970
})-[():WORKED_ON {
    movie = m.title,
    since = m.released,
    sameGeneration = a.born < 1970 AND b.born < 1970
}]->(y = (m):Film {
    title = m.title
})<-[():WORKED_ON {
    movie = m.title
}]-(z = (b):Actor {
    name = b.name,
    experienced = b.born < 1970
}),
''')



rule_supervisor = Rule('''
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)<-[:ACTED_IN]-(z:Person)
WHERE id(x) <> id(z)
GENERATE
((x):Actor {
    name = x.name
})-[(x,z):ACTED_WITH {
    CommonMovies = [y.name]
}]->((z):Actor {
    name = z.name
})
''')

rule = Rule('''
MATCH (x:Person)-[:ACTED_IN]->(y:Movie)<-[:ACTED_IN]-(z:Person)
WHERE id(x) <> id(z)
GENERATE
(a = (x):Actor {
    name = x.name
}),
(b = (z):Actor {
    name = z.name
}),
(a)-[((a),(b)):ACTED_WITH {
    commonMovie = y.title
}]->(b)
''')

# rule10 = Rule('''
# MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person)
# GENERATE
# (x = (a):Actor { name = a.name })-[():ACTED_IN { movie = m.title }]->(y = (m):Film { title = m.title })
# ''')


self._dict:  {'lhs': 'MATCH (m:Movie)', 'constructors': [{'alias': 'x', 'ids': ['x', 'z'], 'labels': ['Film'], 'properties': [{'key': 'title', 'value': 'm.title\n'}]}]}
self._dict:  {'lhs': 'MATCH (a:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(b:Person)', 'constructors': [{'src': {'alias': 'x', 'ids': ['a'], 'labels': ['Actor'], 'properties': [{'key': 'name', 'value': 'a.name'}, {'key': 'experienced', 'value': 'a.born < 1970\n'}]}, 'edge': {'ids': [], 'labels': ['WORKED_ON'], 'properties': [{'key': 'movie', 'value': 'm.title'}, {'key': 'since', 'value': 'm.released'}, {'key': 'sameGeneration', 'value': 'a.born < 1970 AND b.born < 1970\n'}]}, 'tgt': {'alias': 'y', 'ids': ['m'], 'labels': ['Film'], 'properties': [{'key': 'title', 'value': 'm.title\n'}]}}, {'src': {'alias': 'z', 'ids': ['b'], 'labels': ['Actor'], 'properties': [{'key': 'name', 'value': 'b.name'}, {'key': 'experienced', 'value': 'b.born < 1970\n'}]}, 'edge': {'ids': [], 'labels': ['WORKED_ON'], 'properties': [{'key': 'mov

### Execute Rules

In [11]:
my_transform = Transformation([rule])
my_transform.apply_on(graph)

Index: Added 1 index, completed after 31 ms.


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=2, column=7, offset=70>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 70, 'line': 2, 'column': 7}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (x:Person)-[:ACTED_IN]->(y:Movie)<-[:ACTED_IN]-(z:Person)\nWHERE id(x) > id(z)\nMERGE (a:_dummy {\n    _id: "(" + elementID(x) + ")" \n})\nON CREATE\n    SET a:Actor,\n        a.name = x.name\n\nON MATCH\n    SET a:Actor,\n        a.name = \n        CASE\n            WHEN a.name <> x.name\n THEN\n                "Conflict Detected!"\n    

Rule: Added 204 labels, created 102 nodes, set 1653 properties, created 362 relationships, completed after 833 ms.


833

### Abort Transformation

In [10]:
my_transform.abort()

TransformationDeactivationError: This transformation is not currently active.

_id: "(:ACTED_WITH:,(4:84e4e4e9-0f9c-460f-b3c8-bd2685a5e5b6:120),(4:84e4e4e9-0f9c-460f-b3c8-bd2685a5e5b6:122))"

Carrie-Anne Moss _id: "(4:84e4e4e9-0f9c-460f-b3c8-bd2685a5e5b6:120)"

Hugo Weaving _id: "(4:84e4e4e9-0f9c-460f-b3c8-bd2685a5e5b6:122)"